In [19]:
using LowLevelFEM, LinearAlgebra

In [20]:
openGeometry("tarto.geo")

In [21]:
mat = Material("part", E=2e5, ν=0.3)
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [22]:
K = ∫(SymGrad(U)' ⋅ D(:Solid, mat) ⋅ SymGrad(U));

In [23]:
f = ∫(U ⋅ [0, -5, 0], Γ="hole2");

In [24]:
showDoFResults(f, name="f")

0

In [25]:
supp1 = BoundaryCondition("base", uz=0)
supp2 = BoundaryCondition("hole1", ux=0, uy=0)
supp3 = BoundaryCondition("perim", uz=-0.00001);

In [26]:
u = solveField(K, f, support=[supp1, supp2, supp3]);

In [27]:
showDoFResults(u, name="u", visible=true, factor=1);

In [28]:
ε = (u ∘ ∇ + ∇ ∘ u) / 2
showStrainResults(ε, name="ε");
showStrainResults(ε, :z, name="εz");

In [29]:
E = mat.E
ν = mat.ν
I = unitTensor(ε)

σ = E / (1+ν) * (ε + ν/(1-2ν) * trace(ε) * I)
showDoFResults(σ, name="σ");

In [30]:
λ, Q = eigen(σ);

In [31]:
λ1 = λ[1]
λ2 = λ[2]
λ3 = λ[3];

In [32]:
σ_Mohr = λ1 - λ3
showElementResults(σ_Mohr, name="Mohr");

In [33]:
σ_HMH = √(((λ1 - λ2)^2 + (λ2 - λ3)^2 + (λ3 - λ1)^2) / 2)
showElementResults(σ_HMH, name="von Mises");

In [34]:
showElementResults(σ, :z, name="σz")
showElementResults(σ_Mohr - σ_HMH, name="diff")
showDoFResults((σ_Mohr - σ_HMH) / σ_Mohr * 100, name="rel. diff - smooth")

9

In [35]:
b = -σ⋅∇
showElementResults(b, name="b")

10

In [36]:
openPostProcessor()